# Experiment 9C: Clean Held-Out Evaluation

Re-run the standard aggregated-data experiment after verifying no contamination (Exp A).
Same protocol as NB01/NB05 — train on aggregated data (excl FPB), evaluate on FPB.
Trains both BERT-base and ModernBERT-base with LoRA r=16.

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn peft accelerate transformers sentencepiece protobuf

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, training_args
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import json
import gc
import matplotlib.pyplot as plt
import seaborn as sns

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
FPB_SOURCE = 5

label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

remove_cols = [c for c in ds["train"].column_names if c not in ("text", "labels")]
ds = ds.map(
    lambda ex: {
        "text": ex["text"],
        "labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]],
    },
    remove_columns=remove_cols,
)

print(f"Train: {len(ds['train']):,}  |  Val: {len(ds['validation']):,}  |  Test: {len(ds['test']):,}")

def load_fpb_file(agree_level):
    from huggingface_hub import hf_hub_download
    import zipfile, os
    path = hf_hub_download("financial_phrasebank", "data/FinancialPhraseBank-v1.0.zip", repo_type="dataset")
    extract_dir = "/tmp/fpb_data"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(extract_dir)
    filename = {"50agree": "Sentences_50Agree.txt", "allagree": "Sentences_AllAgree.txt"}[agree_level]
    filepath = os.path.join(extract_dir, "FinancialPhraseBank-v1.0", filename)
    sentences, labels = [], []
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            at_idx = line.rfind("@")
            if at_idx == -1:
                continue
            label_str = line[at_idx+1:].strip().lower()
            if label_str in label_map:
                sentences.append(line[:at_idx].strip())
                labels.append(label_map[label_str])
    return {"sentence": sentences, "label": labels}

def load_fpb_dataset(agree_level="50agree"):
    try:
        config = f"sentences_{agree_level}"
        return load_dataset("financial_phrasebank", config, trust_remote_code=True)["train"]
    except Exception:
        return load_fpb_file(agree_level)

fpb_50 = load_fpb_dataset("50agree")
fpb_all = load_fpb_dataset("allagree")
print(f"FPB 50agree: {len(fpb_50['sentence']):,}  |  FPB allAgree: {len(fpb_all['sentence']):,}")

In [ ]:
label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

remove_cols = [c for c in ds["train"].column_names if c not in ("text", "labels")]
ds = ds.map(
    lambda ex: {
        "text": ex["text"],
        "labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]],
    },
    remove_columns=remove_cols,
)

print(f"Train: {len(ds['train']):,}  |  Val: {len(ds['validation']):,}  |  Test: {len(ds['test']):,}")

fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
print(f"FPB 50agree: {len(fpb_50):,}  |  FPB allAgree: {len(fpb_all):,}")

## 2. Evaluation Helper

In [ ]:
def evaluate_on_fpb(model, tokenizer, fpb_dataset, batch_size=32):
    fpb_texts = fpb_dataset["sentence"]
    fpb_labels = fpb_dataset["label"]
    all_preds = []

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(fpb_texts), batch_size), desc="Evaluating"):
            batch = fpb_texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)

    y_true = np.array(fpb_labels)
    y_pred = np.array(all_preds)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    report = classification_report(y_true, y_pred, target_names=LABEL_NAMES)
    cm = confusion_matrix(y_true, y_pred)

    print(f"Accuracy: {acc:.4f} ({int(acc * len(y_true))}/{len(y_true)})")
    print(f"Macro F1: {macro_f1:.4f}")
    print(report)

    return {"accuracy": acc, "macro_f1": macro_f1, "cm": cm, "y_true": y_true, "y_pred": y_pred}

## 3. Train & Evaluate BERT-base + LoRA

In [ ]:
print("Training BERT-base-uncased + LoRA r=16...")
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=NUM_CLASSES)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

bert_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["query", "key", "value", "dense"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.SEQ_CLS,
)
bert_model = get_peft_model(bert_model, bert_lora)
bert_model.gradient_checkpointing_enable()
bert_model = bert_model.cuda()
bert_model.print_trainable_parameters()

def tokenize_bert(examples):
    return bert_tokenizer(examples["text"], truncation=True, max_length=512)

train_data = ds["train"].map(tokenize_bert, batched=True)
val_data = ds["validation"].map(tokenize_bert, batched=True)

bert_trainer = Trainer(
    model=bert_model,
    processing_class=bert_tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=TrainingArguments(
        output_dir="trainer_output_09c_bert",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        fp16=True,
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=2e-4,
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        num_train_epochs=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        gradient_checkpointing=True,
        report_to="none",
        save_total_limit=2,
    ),
    compute_metrics=lambda eval_pred: {
        "accuracy": accuracy_score(
            eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
        )
    },
)

bert_trainer.train()
bert_model = bert_model.cuda().eval()

In [ ]:
print("\n" + "=" * 60)
print("BERT-base + LoRA — FPB sentences_50agree")
print("=" * 60)
bert_50 = evaluate_on_fpb(bert_model, bert_tokenizer, fpb_50)

print("\n" + "=" * 60)
print("BERT-base + LoRA — FPB sentences_allAgree")
print("=" * 60)
bert_all = evaluate_on_fpb(bert_model, bert_tokenizer, fpb_all)

del bert_model, bert_trainer
gc.collect()
torch.cuda.empty_cache()

## 4. Train & Evaluate ModernBERT-base + LoRA

In [ ]:
print("Training ModernBERT-base + LoRA r=16...")
mb_model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=NUM_CLASSES)
mb_tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

mb_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.SEQ_CLS,
)
mb_model = get_peft_model(mb_model, mb_lora)
mb_model.gradient_checkpointing_enable()
mb_model = mb_model.cuda()
mb_model.print_trainable_parameters()

def tokenize_mb(examples):
    return mb_tokenizer(examples["text"], truncation=True, max_length=512)

train_data_mb = ds["train"].map(tokenize_mb, batched=True)
val_data_mb = ds["validation"].map(tokenize_mb, batched=True)

mb_trainer = Trainer(
    model=mb_model,
    processing_class=mb_tokenizer,
    train_dataset=train_data_mb,
    eval_dataset=val_data_mb,
    args=TrainingArguments(
        output_dir="trainer_output_09c_modernbert",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        fp16=True,
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=2e-4,
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        num_train_epochs=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        gradient_checkpointing=True,
        report_to="none",
        save_total_limit=2,
    ),
    compute_metrics=lambda eval_pred: {
        "accuracy": accuracy_score(
            eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
        )
    },
)

mb_trainer.train()
mb_model = mb_model.cuda().eval()

In [ ]:
print("\n" + "=" * 60)
print("ModernBERT-base + LoRA — FPB sentences_50agree")
print("=" * 60)
mb_50 = evaluate_on_fpb(mb_model, mb_tokenizer, fpb_50)

print("\n" + "=" * 60)
print("ModernBERT-base + LoRA — FPB sentences_allAgree")
print("=" * 60)
mb_all = evaluate_on_fpb(mb_model, mb_tokenizer, fpb_all)

del mb_model, mb_trainer
gc.collect()
torch.cuda.empty_cache()

## 5. Comparison Table

In [ ]:
print("=" * 70)
print("EXPERIMENT 9C: CLEAN HELD-OUT COMPARISON")
print("=" * 70)
print(f"{'Model':<30} {'FPB 50agree':>12} {'FPB allAgree':>12}")
print("-" * 60)
print(f"{'BERT-base + LoRA r=16':<30} {bert_50['accuracy']:>11.2%} {bert_all['accuracy']:>12.2%}")
print(f"{'ModernBERT + LoRA r=16':<30} {mb_50['accuracy']:>11.2%} {mb_all['accuracy']:>12.2%}")
print(f"{'Gap (BERT - MB)':<30} {(bert_50['accuracy']-mb_50['accuracy'])*100:>10.2f}pp {(bert_all['accuracy']-mb_all['accuracy'])*100:>10.2f}pp")
print()
print("Previous results (NB01/NB05):")
print(f"  BERT (NB05):      95.19% / 99.16%")
print(f"  ModernBERT (NB01): 91.19% / 99.03%")

results_09c = {
    "bert_50agree": bert_50["accuracy"],
    "bert_allagree": bert_all["accuracy"],
    "bert_50_f1": bert_50["macro_f1"],
    "mb_50agree": mb_50["accuracy"],
    "mb_allagree": mb_all["accuracy"],
    "mb_50_f1": mb_50["macro_f1"],
}
with open("results_09c.json", "w") as f:
    json.dump(results_09c, f, indent=2)
print("\nSaved to results_09c.json")

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (res, title) in zip(axes, [
    (bert_50, "BERT-base + LoRA r=16"),
    (mb_50, "ModernBERT + LoRA r=16"),
]):
    sns.heatmap(
        res["cm"], annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
    )
    ax.set_title(f"{title}\nAcc={res['accuracy']:.2%}  F1={res['macro_f1']:.2%}")
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")

plt.suptitle("Exp 9C: Clean Held-Out — FPB sentences_50agree", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("09c_clean_holdout.png", dpi=150, bbox_inches="tight")
plt.show()